In [1]:
# Create movies.csv
movies_data = """Title,Year,Genre,Rating
The Matrix ,1999,Action|Sci-Fi,8.7
Inception, 2010 ,Sci-Fi| Action , 8.8
Fight Club,1999,Drama ,8.8
"""

with open('movies.csv', 'w') as f:
    f.write(movies_data)

# Create students.csv
students_data = """Name,Subject,Marks
Alice,Math,88
Bob,Science,72
Charlie,Math,95
"""

with open('students.csv', 'w') as f:
    f.write(students_data)

print("CSV files created: movies.csv and students.csv")


CSV files created: movies.csv and students.csv


In [6]:
text_data = """Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and systems to extract knowledge and insights from noisy, structured and unstructured data. Data science is related to data mining, machine learning and big data."""

with open('article.txt', 'w', encoding='utf-8') as f:
    f.write(text_data)

print("File 'article.txt' created.")


File 'article.txt' created.


In [3]:
import pandas as pd
import string
from collections import Counter

# 1) Cleaning movies.csv and generating genre_stats.txt

def clean_movies(input_file='movies.csv', cleaned_file='movies_clean.csv', stats_file='genre_stats.txt'):
    # Read CSV
    df = pd.read_csv(input_file)
    
    # Strip whitespace from all string columns
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Convert Year and Rating to numeric
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
    
    # Split Genre into multiple columns
    genres_split = df['Genre'].str.split('|', expand=True)
    genres_split.columns = [f'Genre{i+1}' for i in range(genres_split.shape[1])]
    
    # Combine original df without Genre + new genre columns
    df_clean = pd.concat([df.drop(columns=['Genre']), genres_split], axis=1)
    
    # Save cleaned movies dataset
    df_clean.to_csv(cleaned_file, index=False)
    
    # Calculate average rating per genre
    # We need to melt the genre columns to get all genres in one column
    genre_cols = genres_split.columns.tolist()
    melted = df_clean.melt(id_vars=['Title', 'Year', 'Rating'], value_vars=genre_cols, value_name='Genre')
    melted = melted.dropna(subset=['Genre'])
    
    genre_avg = melted.groupby('Genre')['Rating'].mean().sort_values(ascending=False)
    
    # Write to genre_stats.txt
    with open(stats_file, 'w') as f:
        for genre, avg_rating in genre_avg.items():
            f.write(f"{genre},{avg_rating:.2f}\n")
            
    print(f"Movies cleaned and saved to {cleaned_file}")
    print(f"Genre stats saved to {stats_file}")
clean_movies()

Movies cleaned and saved to movies_clean.csv
Genre stats saved to genre_stats.txt


In [4]:



# 2) Processing students.csv

def process_students(input_file='students.csv', output_file='summary.csv'):
    df = pd.read_csv(input_file)
    
    # Title case the names
    df['Name'] = df['Name'].str.strip().str.title()
    df['Subject'] = df['Subject'].str.strip()
    
    # Group by subject to get average and highest scorer
    avg_marks = df.groupby('Subject')['Marks'].mean()
    
    # Get topper info for each subject
    toppers = df.loc[df.groupby('Subject')['Marks'].idxmax()]
    
    # Prepare output dataframe
    summary_df = pd.DataFrame({
        'Subject': avg_marks.index,
        'Average': avg_marks.values,
        'Topper': toppers['Name'].values,
        'TopperMarks': toppers['Marks'].values
    })
    
    summary_df.to_csv(output_file, index=False)
    print(f"Student summary saved to {output_file}")
process_students()

Student summary saved to summary.csv


In [7]:



# 3) Text processing of article/chapter

def process_text(input_file='article.txt', output_file='output.txt'):
    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()
        
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Lowercase
    text = text.lower()
    
    # Tokenize words
    words = text.split()
    
    # Count frequency
    freq = Counter(words)
    
    # Get top 20 most common
    top_20 = freq.most_common(20)
    
    # Write to output file
    with open(output_file, 'w', encoding='utf-8') as f:
        for word, count in top_20:
            f.write(f"{word},{count}\n")
    
    print(f"Top 20 words written to {output_file}")

process_text()



Top 20 words written to output.txt
